# Colab track — no Docker, no GPU, no live OLMo

This is the **demo** copy of `T1-06_intervention_map.ipynb`. Replies that would have come from OLMo/Phi are **scripted** (`[OLMo demo]` / `[Phi-4-mini demo]`). Days 2–3 already work without a model.

The live-hosting course is `tier1/` + Docker (or [03_own_environment.md](https://github.com/justkrismanohar/fine-tuning-build/blob/main/03_own_environment.md)).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justkrismanohar/fine-tuning-build/blob/main/colab/day2/T1-06_intervention_map.ipynb)

**Runtime → CPU.** Then **Runtime → Run all**. First code cell clones the course (or finds it if you already have the folder).


In [ ]:
# [RUN] COLAB TRACK — find or clone the course, stay on demo replies
import os
import sys
import subprocess
from pathlib import Path

os.environ["CIVICDESK_COLAB"] = "1"
os.environ["CIVICDESK_MODE"] = "demo"

def _course_root() -> Path:
    markers = Path("tier1") / "day1" / "civicdesk_lab.py"
    for base in [Path("/content/fine-tuning-build"), Path.cwd(), *Path.cwd().resolve().parents]:
        if (base / markers).is_file():
            return base
    dest = Path("/content/fine-tuning-build")
    url = "https://github.com/justkrismanohar/fine-tuning-build.git"
    print("Cloning", url)
    subprocess.check_call(["git", "clone", "--depth", "1", url, str(dest)])
    return dest

ROOT = _course_root()
DAY = ROOT / "tier1" / "day2"
os.chdir(DAY)
if str(DAY) not in sys.path:
    sys.path.insert(0, str(DAY))
if "day2" == "day3":
    d2 = str(ROOT / "tier1" / "day2")
    if d2 not in sys.path:
        sys.path.insert(0, d2)
print("COLAB TRACK")
print(" course:", ROOT)
print(" cwd   :", Path.cwd())
print(" Live OLMo/Phi are OFF. Later chat cells use canned CivicDesk replies.")

# Markdown images in Colab resolve from /content, not cwd
import shutil
content = Path("/content") if Path("/content").is_dir() else Path.cwd()
for folder in ("images", "cards"):
    src = DAY / folder
    dest = content / folder
    if src.is_dir() and dest.resolve() != src.resolve():
        shutil.copytree(src, dest, dirs_exist_ok=True)
        print(f" {folder} →", dest)

# Day 2 helper lives here; Day 3 notebooks that import rag also add ../day2


# T1-06 Intervention map (group)

**Time:** 09:00–09:20 &nbsp;|&nbsp; **Evidence:** a label on last night’s canvas (`cards/intervention_map.md`)

**Notebook learning outcomes**

- Name five interventions: **prompt**, **RAG**, **integration**, **fine-tune**, **none**.
- Pick one for a CivicDesk problem and say why the others are the wrong first move.
- Point at which ministry *type* that problem stands in for — not a live system.

**This block is (group).** Same tables as T1-05. Each member still has this notebook open.


**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the teaching note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** run all cells. Change only lines marked `# [CHANGE]`.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path. Do not drive someone else's notebook.

Git is not required. If you are stuck, load the matching file in `completed/` for this notebook only, mark it as loaded from reference, and continue.


## Overnight readers — what “intervention” means

Day 1 ended with a canvas and a *guess*: prompt / RAG / fine-tune / none. This morning you have to **defend a label**.

An **intervention** is what you change so the next CivicDesk answer is safer or more useful. It is not “buy an LLM.” Five labels cover almost every EOI in the room:

| Label | You change… | You do **not** change… |
|---|---|---|
| **prompt** | The standing instruction (`role: system`) and maybe a few examples in the messages | Weights, index, live systems |
| **RAG** | Which **approved paragraph** is copied into the prompt this turn | Weights. The handbook can be updated tomorrow without retraining |
| **integration** | A **query** to a register, GLPI, GIS, or ticket store | The model is not the source of truth |
| **fine-tune** | A **small adapter** on frozen weights (Day 3: inspect, do not train) | The whole foundation model. Not a substitute for a handbook or a database |
| **none** | The work stays with a person, a form, or a non-LM tool | Do not force a chatbot onto GIS tracks or a password reset |

**Extra words (not on the Day 1 canvas, still needed):**

- **Grounding** — the reply is tied to text you can point at. RAG grounds. Prompt-only often does not.
- **Hallucination** — fluent text with no source. Day 1 already saw invented circulars. T1-09 will pair it with RAG.
- **Source of truth** — where a correct answer *lives*. For serials it is a register (N10). For C2 it is a published handbook (N1). For “reset the account” it is a human (N6).

![Which intervention?](images/t2_intervention_map.jpg)


## Ministry link — pick the row for your table

| Cluster | Sit with | First-move pressure |
|---|---|---|
| **A Criteria** | Labour, Health | Prompt (“never hire / never publish a grade”) plus later RAG to **quote** C2 or N1. Fine-tune is a *voice*, not the rulebook. |
| **B Retrieval** | Finance, Defence OPR, MPSDFS policy, MOWI ECMS, RDLG minutes, THA | RAG. Do not fine-tune a model to memorise every circular. |
| **C Service desk** | Trade IMSD, Defence assets-as-query | Prompt + RAG for KB. **Integration** for serials. **None** for password reset. |
| **D Local control** | CPO, TTDF | Where the box lives is Day 1/3. Many CPO “train on judgments” stories are RAG + custody, not a LoRA on day one. TTDF patrol: **none** (AUX-05). |

This notebook is **N1–N3, N10, N15** as a map. You will *build* RAG in T1-08, not here.


## 1. Six CivicDesk problems — read before you score

Each row is fiction. Gold labels are teaching answers. Argue at the table before you look at `GUESSES`.


In [ ]:
# [RUN]
import json
import civicdesk_rag as rag

print("Data ban: CivicDesk fiction only. No ministry files in a cell.\n")
for c in rag.load_intervention_cases():
    print(f"{c['id']}  [{c['stands_in_for']}]")
    print("   ", c["problem"])
    print()


### Teaching note

**IC-01** looks like “an LLM that knows circulars.” The paragraph already exists. That is **RAG**. Fine-tuning would freeze yesterday’s wording.

**IC-02** is Day 1 again. The visitor did not change. The wall instruction did. **prompt**.

**IC-03** is **integration** (or **none** if you refuse to guess serials). T1-05 named this: the model is not the register.

**IC-04** is the only honest **fine-tune** in the set: a *stable phrasing* (“quote, flag equivalent, do not hire”). The C2 *text* can still move; that part stays RAG.

**IC-05** and **IC-06** are **none**. One is operational data. One is a privileged act. A fluent chatbot is the wrong object.

If your overnight canvas said “an LLM” with no label, pick one now. Unsure is allowed only with a question attached.


## 2. Score the map

`# [CHANGE]` only if you want to test yourself with different labels. Allowed strings: `prompt`, `RAG`, `integration`, `fine-tune`, `none`.


In [ ]:
# [CHANGE] try your table's labels first; this cell ships the teaching key so Run All works.
GUESSES = {
    "IC-01": "RAG",
    "IC-02": "prompt",
    "IC-03": "integration",
    "IC-04": "fine-tune",
    "IC-05": "none",
    "IC-06": "none",
}

import civicdesk_rag as rag

rag.score_interventions(GUESSES)
rag.save_log("T1-06_map", {"guesses": GUESSES})


### After you run

Copy one line onto `cards/intervention_map.md`: **your T1-05 guess → your T1-06 label**. If they differ, that is the learning.

**Write-up (one sentence):** which intervention will *your* cluster practise this afternoon, and which one must you *not* pretend to have built today?


In [ ]:
# [OPTIONAL] deep-code: count how many EOI types in the handbook metadata are retrieval-shaped.
import civicdesk_rag as rag
from collections import Counter

c = Counter()
for ch in rag.load_chunks():
    for s in ch.get("stands_in_for", []):
        c[s.split("(")[0].strip()] += 1
print("chunk mentions (not a ministry ranking):")
for k, n in c.most_common():
    print(f"  {n:2}  {k}")
